# T2.L6. Системи комп'ютерної математики та їх можливості для математичного моделювання

Демонстраційний notebook: **SymPy → analytical model → lambdify → SciPy verification → sensitivity**.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sympy as sp

lesson = Path.cwd()
if lesson.name == "notebooks":
    lesson = lesson.parent
sys.path.insert(0, str(lesson / "src"))

from model import (
    symbolic_model, equilibrium, analytical_solution, lambdified_solution,
    numerical_solution, cumulative_state, threshold_time,
    max_symbolic_numeric_error
)

## 1. Символьна постановка

\[
\frac{dS}{dt}=q-kS,\qquad S(0)=S_0.
\]

In [ ]:
sm = symbolic_model()
sm["ode"], sm["equilibrium"], sm["solution"], sm["residual"]

`residual = 0` означає, що отриманий вираз символічно задовольняє ODE.

In [ ]:
print("Integral:", sm["cumulative"])
print("dS*/dq =", sm["sensitivity_q"])
print("dS*/dk =", sm["sensitivity_k"]) 

## 2. Baseline

In [ ]:
q, k, s0 = 12.0, 0.10, 20.0
print("Equilibrium:", equilibrium(q, k))
print("S(10):", analytical_solution(10.0, q, k, s0))
print("Integral 0..10:", cumulative_state(10.0, q, k, s0))
print("t for S=80:", threshold_time(80.0, q, k, s0))

## 3. SymPy → NumPy через lambdify

In [ ]:
times = np.linspace(0, 30, 121)
closed = analytical_solution(times, q, k, s0)
fn = lambdified_solution()
from_sympy = fn(times, q, k, s0)
np.max(np.abs(closed - from_sympy))

## 4. Незалежна SciPy verification

In [ ]:
numeric = numerical_solution(times, q, k, s0)
error = np.max(np.abs(closed - numeric))
print("max abs error =", error)

plt.figure(figsize=(8, 4.5))
plt.plot(times, closed, label="analytical")
plt.plot(times, from_sympy, "--", label="lambdify")
plt.plot(times, numeric, ":", label="solve_ivp")
plt.axhline(equilibrium(q, k), linestyle="--", linewidth=1, label="equilibrium")
plt.xlabel("t")
plt.ylabel("S(t)")
plt.title("Symbolic and numerical representations of the same model")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 5. Параметричний експеримент

In [ ]:
ks = [0.05, 0.08, 0.10, 0.12, 0.15]
sens = pd.DataFrame({
    "k": ks,
    "equilibrium": [equilibrium(q, x) for x in ks],
    "S_t10": [analytical_solution(10, q, x, s0) for x in ks],
})
sens

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(sens["k"], sens["equilibrium"], marker="o", label="S*")
plt.plot(sens["k"], sens["S_t10"], marker="o", label="S(10)")
plt.xlabel("k")
plt.ylabel("state")
plt.title("Sensitivity to loss coefficient k")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

## 6. Інтерпретація

Символьна похідна

\[
\frac{\partial S^*}{\partial k}=-\frac{q}{k^2}<0
\]

наперед показує напрям впливу параметра. Чисельний експеримент підтверджує цей структурний висновок для обраних значень \(k\).

**Research transfer:** яку залежність у власному дослідженні доцільно спочатку дослідити символьно, а потім перевірити чисельно?